**11장 – 심층신경망 훈련하기**


# 복습 & 데이터 증식(Data Augmentation)

## MNIST Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

from keras.datasets import mnist     # MNIST dataset is included in Keras
from keras.models import Sequential  # Model type to be used

from keras.layers import Dense, Dropout, Activation, Input # Types of layers to be used in our model
from keras.utils import to_categorical

MNIST 데이터를 읽어서 트레인 데이터와 테스트 데이터로 각각 로딩한다.

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

In [ ]:
idx = 100
plt.imshow(X_train[idx], cmap='gray', interpolation='none')
plt.title("Class {}".format(y_train[idx]))


In [ ]:
print(y_train[idx])

## Training

### 신경망 준비하기

In [ ]:
X_train = X_train.reshape(60000, 784) # reshape 60,000 28 x 28 matrices into 60,000 784-length vectors.
X_test = X_test.reshape(10000, 784)   # reshape 10,000 28 x 28 matrices into 10,000 784-length vectors.

X_train = X_train.astype('float32')   # change integers to 32-bit floating point numbers
X_test = X_test.astype('float32')

X_train /= 255                        # normalize each value for each pixel for the entire vector for each input
X_test /= 255

print("Training matrix shape", X_train.shape)
print("Testing matrix shape", X_test.shape)

In [ ]:
nb_classes = 10 # number of unique digits
#to_categorical converts the labeled data to one-hot vector

Y_train = to_categorical(y_train, nb_classes)
Y_test = to_categorical(y_test, nb_classes)
Y_train

In [ ]:
model = Sequential([
    Input(shape=(784,)),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(10, activation='softmax')
])

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
model.summary()

### 학습

In [ ]:
model.fit(X_train, Y_train,
          batch_size=128, epochs=20,
          verbose=1)

학습된 모델이 얼마나 정확한지 평가 해보자

In [ ]:
score = model.evaluate(X_test, Y_test)
print('Test score:', score[0])
print('Test accuracy:', score[1])

## See if it recognizes my handwriting

#### 글씨를 쓸수 있는 작은 웹앱을 만들자

In [ ]:
from IPython.display import HTML
from google.colab.output import eval_js
from base64 import b64decode

canvas_html = """
<canvas width=%d height=%d></canvas>
<button>Finish</button>
<script>
var canvas = document.querySelector('canvas')
var ctx = canvas.getContext('2d')
ctx.lineWidth = %d
var button = document.querySelector('button')
var mouse = {x: 0, y: 0}

canvas.addEventListener('mousemove', function(e) {
  mouse.x = e.pageX - this.offsetLeft
  mouse.y = e.pageY - this.offsetTop
})
canvas.onmousedown = ()=>{
  ctx.beginPath()
  ctx.moveTo(mouse.x, mouse.y)
  canvas.addEventListener('mousemove', onPaint)
}
canvas.onmouseup = ()=>{
  canvas.removeEventListener('mousemove', onPaint)
}
var onPaint = ()=>{
  ctx.lineTo(mouse.x, mouse.y)
  ctx.stroke()
}

var data = new Promise(resolve=>{
  button.onclick = ()=>{
    resolve(canvas.toDataURL('image/png'))
  }
})
</script>
"""

def draw(filename='drawing.png', w=280, h=280, line_width=20):
  display(HTML(canvas_html % (w, h, line_width)))
  data = eval_js("data")
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return len(binary)

#### 마우스로 숫자를 써서 인식시켜보자

In [ ]:
from PIL import Image
import numpy as np
#from IPython.display import Image
def predict_my_drawing(model):
    draw()
    image = Image.open('drawing.png')

    # summarize some details about the image
    print(image.format, image.size, image.mode)
    image = image.resize((28,28))
    image = image.convert('LA')

    np_frame = np.array(image.getdata())
    digit = np_frame[:,1].reshape(28,28)
    digit_n = digit / 255
    reco = model.predict([digit_n.reshape(-1,784)])[0].reshape(10,-1)
    print(reco)
    print("recognized as", np.argmax(reco))
    return digit

In [ ]:
digit=predict_my_drawing(model)

### Data Augmentation

In [ ]:
import numpy as np
from keras.datasets import mnist
from torchvision import transforms
from PIL import Image

# Load MNIST dataset from Keras
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Define the transformations: rotation and translation (relocation)
transform = transforms.Compose([
    transforms.RandomRotation(degrees=(-30, 30)),  # Rotate between -30 to 30 degrees
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1))  # Relocate by translating up to 10%
])

# Function to apply the transformation on an array
def apply_transform(image_array):
    # Convert the NumPy array to a PIL image (grayscale mode)
    pil_image = Image.fromarray(image_array)
    # Apply the transformations
    transformed_image = transform(pil_image)
    # Convert the transformed image back to NumPy array
    return np.array(transformed_image)

# Apply the transformation to all images in X_train
X_train_transformed = np.array([apply_transform(img) for img in X_train])


In [ ]:
idx=90
plt.subplot(1,2,1)
plt.imshow(X_train[idx], cmap='gray', interpolation='none')
plt.subplot(1,2,2)
plt.imshow(X_train_transformed[idx], cmap='gray', interpolation='none')

In [ ]:
X_train = np.concatenate((X_train, X_train_transformed), axis = 0)
y_train = np.concatenate((y_train, y_train), axis = 0)

In [ ]:
X_train = X_train.reshape(120000, 784) # reshape 60,000 28 x 28 matrices into 60,000 784-length vectors.
X_test = X_test.reshape(10000, 784)   # reshape 10,000 28 x 28 matrices into 10,000 784-length vectors.

X_train = X_train.astype('float32')   # change integers to 32-bit floating point numbers
X_test = X_test.astype('float32')

X_train /= 255                        # normalize each value for each pixel for the entire vector for each input
X_test /= 255

print("Training matrix shape", X_train.shape)
print("Testing matrix shape", X_test.shape)

In [ ]:
nb_classes = 10 # number of unique digits
#to_categorical converts the labeled data to one-hot vector

Y_train = to_categorical(y_train, nb_classes)
Y_test = to_categorical(y_test, nb_classes)
Y_train

In [ ]:
model2 = Sequential([
    Input(shape=(784,)),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(10, activation='softmax')
])

model2.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
model2.fit(X_train, Y_train,
          batch_size=128, epochs=20,
          verbose=1)

In [ ]:
def compare_my_drawing(model1, model2):
    draw()
    image = Image.open('drawing.png')

    # summarize some details about the image
    print(image.format, image.size, image.mode)
    image = image.resize((28,28))
    image = image.convert('LA')

    np_frame = np.array(image.getdata())
    digit = np_frame[:,1].reshape(28,28)
    digit_n = digit / 255
    reco1 = model1.predict([digit_n.reshape(-1,784)])[0].reshape(10,-1)
    reco2 = model2.predict([digit_n.reshape(-1,784)])[0].reshape(10,-1)
    print("model1")
    print(reco1)
    print("recognized as", np.argmax(reco1))
    print("model2")
    print(reco2)
    print("recognized as", np.argmax(reco2))
    return digit

In [ ]:
digit=compare_my_drawing(model, model2)

# 심층 신경망의 주요 요소에 대한 이해

심층 신경망의 고려 사항

* **까다로운 그레이디언트 소실 또는 그레이디언트 폭주 문제** 심층 신경망의 아래쪽으로 갈수록 그레이디언트가 점점 더 작아지거나 커지는 현상 두 현상 모두 하위층을 훈련하기 매우 어렵다.
* **비용 문제** 대규모 신경망을 위한 훈련 데이터가 충분하지 않거나 레이블을 만드는 작업에 비용이 너무 많이 들 수 있다.
* **훈련이 극단적으로 느려질 수 있다.**
* **과대적합** 수백만 개의 파라미터를 가진 모델은 훈련 세트에 과대적합될 위험이 매우 크다. 특히 훈련 샘플이 충분하지 않거나 잡음이 많을 때.

## 준비작업

In [ ]:
from tensorflow import keras
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
keras.backend.clear_session()

In [ ]:
fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

X_valid, X_train = X_train_full[:5000] / 255., X_train_full[5000:] / 255.
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]
X_test = X_test / 255.

In [ ]:
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

In [ ]:
import matplotlib.pyplot as plt
def plot_loss(*args, figsize=(10,5)):
    def plot_ind_loss(history, label):
        plt.plot(history.history['loss'], label=label)
        plt.title('loss over epochs')
        plt.ylabel('loss')
        plt.xlabel('epochs')
        plt.legend()

    plt.figure(figsize=figsize)
    for i, fstr in enumerate(args):
        f = globals().get(fstr) if isinstance(fstr, str) else fstr
        fstr = fstr if isinstance(fstr, str) else i
        plot_ind_loss(f, fstr)

## Initializer

kernel_initializer를 표시 하지 않으면 글로럿 초기화를 사용하게 된다.

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

history = model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))
print('\n\n**********************************')
model.evaluate(X_test, y_test)

kernel_initializer를 he_normal로 바꾸어 다시 수행 해보자

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, activation="relu", kernel_initializer='he_normal'),
    keras.layers.Dense(100, activation="relu", kernel_initializer='he_normal'),
    keras.layers.Dense(10, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

h_he = model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))
print('\n\n**********************************')
model.evaluate(X_test, y_test)

In [ ]:
plot_loss("history", "h_he")

20회의 epoch과 기타 다른 하이퍼 파라미터의 조합으로 인해 어떤 initializer가 좋은지 결론을 지을수 없지만, 결과가 다른걸 볼수 있다.

참고 : https://keras.io/api/layers/initializers/

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, activation="relu", kernel_initializer=keras.initializers.RandomNormal(mean=0.0, stddev=0.05, seed=None)),
    keras.layers.Dense(100, activation="relu", kernel_initializer=keras.initializers.RandomNormal(mean=0.0, stddev=0.05, seed=None)),
    keras.layers.Dense(10, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

h_random = model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))
print('\n\n**********************************')
model.evaluate(X_test, y_test)

In [ ]:
plot_loss("history", "h_he", "h_random")

## Activation

ReLU, LeakyReLU, ELU, SELU등은 gradient vanishing 혹은 exploding 을 막기 위해 만들어졌다.

Activation을 sigmoid로 바꾸어서 학습을 진행 해보자.

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, activation="sigmoid"),
    keras.layers.Dense(100, activation="sigmoid"),
    keras.layers.Dense(10, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

h_sigmoid = model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))
print('\n\n**********************************')
model.evaluate(X_test, y_test)

LeakyReLU를 사용해 보자.

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300),
    keras.layers.LeakyReLU(alpha=0.2),
    keras.layers.Dense(100),
    keras.layers.LeakyReLU(alpha=0.2),
    keras.layers.Dense(10, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

h_leaky = model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))
print('\n\n**********************************')
model.evaluate(X_test, y_test)

LeakyReLU를 he_normal 초기화와 같이 써보자.

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, kernel_initializer='he_normal'),
    keras.layers.LeakyReLU(alpha=0.2),
    keras.layers.Dense(100, kernel_initializer='he_normal'),
    keras.layers.LeakyReLU(alpha=0.2),
    keras.layers.Dense(10, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

h_leaky_he = model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))
print('\n\n**********************************')
model.evaluate(X_test, y_test)

selu(Scaled Exponential LU)의 경우는 다음과 같이 바로 지정하여 사용 할수 있다.

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, activation="selu", kernel_initializer='lecun_normal'),
    keras.layers.Dense(100, activation="selu", kernel_initializer='lecun_normal'),
    keras.layers.Dense(10, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

h_selu = model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))
print('\n\n**********************************')
model.evaluate(X_test, y_test)

In [ ]:
plot_loss("history", "h_he", "h_random","h_sigmoid", "h_leaky", "h_leaky_he", "h_selu")

In [ ]:
plot_loss("history", "h_he", "h_random", "h_leaky", "h_leaky_he", "h_selu")

## 배치 정규화

이런 새로운 Activation이 그래디언 폭주와 소실을 많이 완화를 시켰지만, 그래도 다시 발생 할 수 있다. 그래서 각층에 배치정규화를 하여 폭주나 소실을 막는다. : 배치정규화


![BatchNormalization](https://drive.google.com/uc?id=16SrkADy5fgHVlZg0S9gHrFREKVBFbVlN)

참고 : https://keras.io/api/layers/regularizers/


In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(300, activation="elu", kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(100, activation="elu", kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(10, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))
print('\n\n**********************************')
model.evaluate(X_test, y_test)

In [ ]:
model.summary()

In [ ]:
[(var.name, var.trainable) for var in model.layers[1].variables]

혹은 Activation 이전에 수행 할수도 있는데, 두가지 다 시행 해보고 원하는 방식으로 하는걸 추천한다.

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(300, kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('elu'),
    keras.layers.Dense(100, kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('elu'),
    keras.layers.Dense(10, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))
print('\n\n**********************************')
model.evaluate(X_test, y_test)

## Optimizer

Momentum

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

optimizer = keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=optimizer,
              metrics=["accuracy"])

model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))
print('\n\n**********************************')
model.evaluate(X_test, y_test)

NAG(Nesterov Accelerated Gradient)

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

optimizer = keras.optimizers.SGD(learning_rate=0.001, momentum=0.9, nesterov=True)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=optimizer,
              metrics=["accuracy"])

model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))
print('\n\n**********************************')
model.evaluate(X_test, y_test)

Adam(Adaptive Moment Estimation)

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=optimizer,
              metrics=["accuracy"])

model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))
print('\n\n**********************************')
model.evaluate(X_test, y_test)

## Dropout

#### Fully Connected Network

In [ ]:
np.random.seed(11)
tf.random.set_seed(11)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, activation="relu", kernel_initializer='he_normal'),
    keras.layers.Dense(100, activation="relu", kernel_initializer='he_normal'),
    keras.layers.Dense(10, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))
print('\n\n**********************************')
model.evaluate(X_test, y_test)

#### Dropout

In [ ]:
np.random.seed(11)
tf.random.set_seed(11)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    #keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(300, activation="relu", kernel_initializer='he_normal'),
    #keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(100, activation="relu", kernel_initializer='he_normal'),
    keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(10, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))
print('\n\n**********************************')
model.evaluate(X_test, y_test)